# 🫁 DCGAN for Chest X-Ray Augmentation — PneumoniaMNIST
**Author:** Amlan Sarkar  
**Goal:** Train a DCGAN to generate synthetic chest X-rays of the minority class, use them to balance the dataset, and prove it improves classifier performance.

### Pipeline
1. Load & analyse PneumoniaMNIST class distribution
2. Build DCGAN (Generator + Discriminator) in PyTorch
3. Train GAN on minority class only
4. Generate synthetic images to fill class gap
5. Train CNN classifier — with vs without GAN augmentation
6. Compare metrics (Accuracy, F1, AUC-ROC)
7. Save all result plots for LinkedIn showcase

In [ ]:
!pip install medmnist scikit-learn 

In [ ]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Subset, ConcatDataset, TensorDataset
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import classification_report, roc_auc_score, f1_score
import os, warnings
warnings.filterwarnings('ignore')

torch.manual_seed(42)
np.random.seed(42)

LATENT_DIM = 100
BATCH_SIZE = 64
GAN_EPOCHS = 300
CLF_EPOCHS = 20
LR         = 0.0002
DEVICE     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
os.makedirs('results_v2', exist_ok=True)
print(f'Using device: {DEVICE}')

## 1. Load Dataset & Analyse Class Distribution

In [ ]:
from medmnist import PneumoniaMNIST

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])
])

train_data = PneumoniaMNIST(split='train', transform=transform, download=True)
test_data  = PneumoniaMNIST(split='test',  transform=transform, download=True)

all_labels = np.array([int(train_data[i][1][0]) for i in range(len(train_data))])

n_normal       = (all_labels == 0).sum()
n_pneumonia    = (all_labels == 1).sum()
minority_class = 0 if n_normal < n_pneumonia else 1
gap            = abs(n_pneumonia - n_normal)

print(f'Total training samples : {len(train_data)}')
print(f'Normal      (class 0)  : {n_normal}')
print(f'Pneumonia   (class 1)  : {n_pneumonia}')
print(f'Minority class         : {minority_class}')
print(f'Gap to fill            : {gap} samples')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(['Normal (0)', 'Pneumonia (1)'], [n_normal, n_pneumonia],
            color=['steelblue', 'tomato'], edgecolor='black')
axes[0].set_title('Class Distribution — PneumoniaMNIST', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Number of Samples')
for i, v in enumerate([n_normal, n_pneumonia]):
    axes[0].text(i, v + 10, str(v), ha='center', fontweight='bold')

sample_idx  = [i for i in range(len(train_data)) if int(train_data[i][1][0]) == minority_class][:8]
sample_imgs = torch.stack([train_data[i][0] for i in sample_idx])
grid = torchvision.utils.make_grid(sample_imgs, nrow=8, normalize=True)
axes[1].imshow(grid.permute(1,2,0).numpy(), cmap='gray')
axes[1].set_title(f'Real Minority Class (class {minority_class}) Samples', fontsize=13, fontweight='bold')
axes[1].axis('off')

plt.tight_layout()
plt.savefig('results/01_class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved -> results/01_class_distribution.png')

## 2. DCGAN Architecture

In [ ]:
def weights_init(m):
    if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d)):
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif isinstance(m, nn.BatchNorm2d):
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)


class Generator(nn.Module):
    """
    Input : (batch, LATENT_DIM, 1, 1) - random noise
    Output: (batch, 1, 28, 28)        - synthetic X-ray
    """
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            # (100,1,1) -> (256,3,3)
            nn.ConvTranspose2d(LATENT_DIM, 256, 3, 1, 0, bias=False),
            nn.BatchNorm2d(256), nn.ReLU(True),
            # (256,3,3) -> (128,7,7)
            nn.ConvTranspose2d(256, 128, 3, 2, 0, bias=False),
            nn.BatchNorm2d(128), nn.ReLU(True),
            # (128,7,7) -> (64,14,14)
            nn.ConvTranspose2d(128, 64, 4, 2, 1, bias=False),
            nn.BatchNorm2d(64), nn.ReLU(True),
            # (64,14,14) -> (1,28,28)
            nn.ConvTranspose2d(64, 1, 4, 2, 1, bias=False),
            nn.Tanh()
        )
    def forward(self, z): return self.net(z)


class Discriminator(nn.Module):
    """
    Input : (batch, 1, 28, 28) - real or fake X-ray
    Output: (batch, 1)         - P(image is real)
    """
    def __init__(self):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 64, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(64, 128, 4, 2, 1, bias=False),
            nn.BatchNorm2d(128), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(128, 256, 3, 2, 0, bias=False),
            nn.BatchNorm2d(256), nn.LeakyReLU(0.2, inplace=True),
        )
        self.pool = nn.AdaptiveAvgPool2d((2, 2))
        self.fc   = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 2 * 2, 1),
            nn.Sigmoid()
        )
    def forward(self, img): return self.fc(self.pool(self.conv(img)))


print('Generator:'); print(Generator())
print('\nDiscriminator:'); print(Discriminator())

In [ ]:
G = Generator().to(DEVICE);     G.apply(weights_init)
D = Discriminator().to(DEVICE); D.apply(weights_init)

with torch.no_grad():
    _z    = torch.randn(4, LATENT_DIM, 1, 1).to(DEVICE)
    g_out = G(_z)
    d_out = D(g_out)
    assert g_out.shape == (4, 1, 28, 28)
    assert d_out.shape == (4, 1)

print(f'Generator output shape   : {g_out.shape}')
print(f'Discriminator output shape: {d_out.shape}')
print('Architecture verified — ready to train!')

## 3. Train DCGAN on Minority Class

In [ ]:
minority_idx = [i for i in range(len(train_data)) if int(train_data[i][1][0]) == minority_class]
minority_subset = Subset(train_data, minority_idx)
minority_loader = DataLoader(minority_subset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)

print(f'Minority class samples : {len(minority_subset)}')
print(f'Batches per epoch      : {len(minority_loader)}')

In [ ]:
criterion = nn.BCELoss()
opt_G = torch.optim.Adam(G.parameters(), lr=LR, betas=(0.5, 0.999))
opt_D = torch.optim.Adam(D.parameters(), lr=LR, betas=(0.5, 0.999))

G_losses, D_losses = [], []

print('Starting GAN training...\n')
for epoch in range(GAN_EPOCHS):
    g_ep, d_ep = 0, 0
    for real_imgs, _ in minority_loader:
        real_imgs   = real_imgs.to(DEVICE)
        b           = real_imgs.size(0)
        real_labels = torch.full((b, 1), 0.9).to(DEVICE)   # label smoothing
        fake_labels = torch.full((b, 1), 0.1).to(DEVICE)   # label smoothing

        # Train Discriminator
        z         = torch.randn(b, LATENT_DIM, 1, 1).to(DEVICE)
        fake_imgs = G(z).detach()
        loss_D    = (criterion(D(real_imgs), real_labels) +
                     criterion(D(fake_imgs), fake_labels))
        opt_D.zero_grad(); loss_D.backward(); opt_D.step()

        # Train Generator
        z      = torch.randn(b, LATENT_DIM, 1, 1).to(DEVICE)
        loss_G = criterion(D(G(z)), real_labels)
        opt_G.zero_grad(); loss_G.backward(); opt_G.step()

        g_ep += loss_G.item(); d_ep += loss_D.item()

    G_losses.append(g_ep / len(minority_loader))
    D_losses.append(d_ep / len(minority_loader))
    if (epoch + 1) % 30 == 0:
        print(f'Epoch [{epoch+1:>3}/{GAN_EPOCHS}]  D Loss: {D_losses[-1]:.4f}  G Loss: {G_losses[-1]:.4f}')

torch.save(G.state_dict(), 'results/dcgan_generator.pth')
print('\nTraining complete. Generator saved -> results/dcgan_generator.pth')


In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(G_losses, label='Generator Loss',     color='tomato',    linewidth=2)
plt.plot(D_losses, label='Discriminator Loss', color='steelblue', linewidth=2)
plt.xlabel('Epoch', fontsize=12); plt.ylabel('Loss', fontsize=12)
plt.title('DCGAN Training Loss — PneumoniaMNIST', fontsize=14, fontweight='bold')
plt.legend(fontsize=11); plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('results/02_training_loss.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved -> results/02_training_loss.png')

## 4. Generate & Visualise Synthetic X-Rays

In [ ]:
G.eval()
with torch.no_grad():
    z           = torch.randn(16, LATENT_DIM, 1, 1).to(DEVICE)
    sample_imgs = G(z).cpu()

grid = torchvision.utils.make_grid(sample_imgs, nrow=8, normalize=True, padding=2)
plt.figure(figsize=(14, 4))
plt.imshow(grid.permute(1, 2, 0).numpy(), cmap='gray')
plt.title(f'DCGAN Generated Synthetic Chest X-Rays (Class {minority_class})', fontsize=14, fontweight='bold')
plt.axis('off'); plt.tight_layout()
plt.savefig('results/03_synthetic_xrays.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved -> results/03_synthetic_xrays.png')

In [ ]:
real_samples = torch.stack([train_data[i][0] for i in minority_idx[:8]])
with torch.no_grad():
    z    = torch.randn(8, LATENT_DIM, 1, 1).to(DEVICE)
    fake = G(z).cpu()

fig, axes = plt.subplots(2, 8, figsize=(16, 5))
fig.suptitle('Real (top) vs GAN-Generated (bottom) Chest X-Rays', fontsize=14, fontweight='bold')
for i in range(8):
    axes[0, i].imshow(real_samples[i].squeeze(), cmap='gray')
    axes[0, i].axis('off'); axes[0, i].set_title('Real', fontsize=9, color='steelblue')
    axes[1, i].imshow(fake[i].squeeze(), cmap='gray')
    axes[1, i].axis('off'); axes[1, i].set_title('Synthetic', fontsize=9, color='tomato')

plt.tight_layout()
plt.savefig('results/04_real_vs_synthetic.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved -> results/04_real_vs_synthetic.png')

## 5. Classifier Experiment — With vs Without GAN Augmentation

In [ ]:
def custom_collate(batch):
    imgs   = torch.stack([item[0] for item in batch])
    labels = torch.tensor(
                [int(item[1][0]) if isinstance(item[1], np.ndarray)
                 else int(item[1]) for item in batch]
             ).unsqueeze(1)
    return imgs, labels


class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((4, 4)), nn.Flatten(),
            nn.Linear(128 * 4 * 4, 256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, 2)
        )
    def forward(self, x): return self.classifier(self.features(x))


def train_classifier(loader, tag='', use_weights=False):
    model   = SimpleCNN().to(DEVICE)
    opt     = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
    if use_weights:
        weight = torch.tensor([
            len(all_labels) / (2 * n_pneumonia),
            len(all_labels) / (2 * n_normal)
        ], dtype=torch.float).to(DEVICE)
        loss_fn = nn.CrossEntropyLoss(weight=weight)
    else:
        loss_fn = nn.CrossEntropyLoss()
    model.train()
    for epoch in range(CLF_EPOCHS):
        total = 0
        for imgs, labels in loader:
            imgs   = imgs.to(DEVICE)
            labels = labels.squeeze().long().to(DEVICE)
            loss   = loss_fn(model(imgs), labels)
            opt.zero_grad(); loss.backward(); opt.step()
            total += loss.item()
        if (epoch + 1) % 5 == 0:
            print(f'  [{tag}] Epoch {epoch+1}/{CLF_EPOCHS}  Loss: {total/len(loader):.4f}')
    return model


def evaluate(model, loader, threshold=0.35):
    """threshold < 0.5 biases toward predicting Normal (class 0)"""
    model.eval()
    preds, probs, true = [], [], []
    with torch.no_grad():
        for imgs, labels in loader:
            out = model(imgs.to(DEVICE))
            p   = torch.softmax(out, 1)[:, 1].cpu().numpy()  # P(Pneumonia)
            probs.extend(p)
            preds.extend((p >= threshold).astype(int))        # 0=Normal if P(Pneumonia)<threshold
            true.extend(labels.squeeze().long().numpy())
    acc = (np.array(preds) == np.array(true)).mean()
    f1  = f1_score(true, preds, average='macro')
    auc = roc_auc_score(true, probs)
    return acc, f1, auc, true, preds

print('SimpleCNN ready.')


In [ ]:
CLF_EPOCHS = 25

print('[1/2] Training on ORIGINAL imbalanced data...')
original_loader = DataLoader(train_data, batch_size=BATCH_SIZE,
                             shuffle=True,  collate_fn=custom_collate)
test_loader     = DataLoader(test_data,  batch_size=BATCH_SIZE,
                             shuffle=False, collate_fn=custom_collate)

clf_original = train_classifier(original_loader, tag='Original', use_weights=True)
acc_orig, f1_orig, auc_orig, _, _ = evaluate(clf_original, test_loader, threshold=0.35)
print(f'\nOriginal -> Accuracy: {acc_orig:.4f} | F1: {f1_orig:.4f} | AUC: {auc_orig:.4f}')


In [ ]:
print(f'Generating quality-filtered synthetic images...')
G.eval(); D.eval()
syn_list, generated = [], 0
FILL_RATIO  = 0.5
target_size = int(gap * FILL_RATIO)
CONFIDENCE_THRESHOLD = 0.4

with torch.no_grad():
    attempts = 0
    while generated < target_size:
        n     = min(BATCH_SIZE * 2, (target_size - generated) * 3)
        z     = torch.randn(n, LATENT_DIM, 1, 1).to(DEVICE)
        imgs  = G(z)
        scores = D(imgs).squeeze()
        mask   = scores > CONFIDENCE_THRESHOLD
        good_imgs = imgs[mask].cpu()
        if len(good_imgs) > 0:
            syn_list.append(good_imgs)
            generated += len(good_imgs)
        attempts += 1
        if attempts > 500: break

syn_imgs    = torch.cat(syn_list)[:target_size]
syn_labels  = torch.full((len(syn_imgs),), minority_class, dtype=torch.long).unsqueeze(1)
syn_dataset = TensorDataset(syn_imgs, syn_labels)

augmented_dataset = ConcatDataset([train_data, syn_dataset])
augmented_loader  = DataLoader(augmented_dataset, batch_size=BATCH_SIZE,
                               shuffle=True, collate_fn=custom_collate)

print(f'Target size    : {target_size}')
print(f'Accepted       : {len(syn_imgs)}')
print(f'Augmented total: {len(augmented_dataset)}')


In [ ]:
print('[2/2] Training on GAN-AUGMENTED balanced data...')
clf_aug = train_classifier(augmented_loader, tag='Augmented')
acc_aug, f1_aug, auc_aug, true_labels, aug_preds = evaluate(clf_aug, test_loader)
print(f'\nAugmented -> Accuracy: {acc_aug:.4f} | F1: {f1_aug:.4f} | AUC: {auc_aug:.4f}')


## 6. Results Comparison

In [ ]:
metrics     = ['Accuracy', 'Macro F1', 'AUC-ROC']
orig_scores = [acc_orig, f1_orig, auc_orig]
aug_scores  = [acc_aug,  f1_aug,  auc_aug]
x, w = np.arange(len(metrics)), 0.3

fig, ax = plt.subplots(figsize=(9, 5))
b1 = ax.bar(x - w/2, orig_scores, w, label='Without GAN Aug', color='steelblue', edgecolor='black', alpha=0.85)
b2 = ax.bar(x + w/2, aug_scores,  w, label='With GAN Aug',    color='tomato',    edgecolor='black', alpha=0.85)
for bar in b1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{bar.get_height():.3f}', ha='center', fontsize=10, fontweight='bold', color='steelblue')
for bar in b2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{bar.get_height():.3f}', ha='center', fontsize=10, fontweight='bold', color='tomato')
ax.set_xticks(x); ax.set_xticklabels(metrics, fontsize=12)
ax.set_ylim(0, 1.15); ax.set_ylabel('Score', fontsize=12)
ax.set_title('Classifier Performance: Original vs GAN-Augmented Data', fontsize=13, fontweight='bold')
ax.legend(fontsize=11); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('results/05_metrics_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved -> results/05_metrics_comparison.png')

In [ ]:
print('=' * 55)
print('           FINAL RESULTS SUMMARY')
print('=' * 55)
print(f'{"Metric":<20} {"Original":>12} {"GAN Aug":>12}')
print('-' * 55)
for name, o, a in zip(metrics, orig_scores, aug_scores):
    print(f'{name:<20} {o:>12.4f} {a:>12.4f}')
print('=' * 55)
print(f'F1 improvement  : +{f1_aug  - f1_orig:.4f}')
print(f'AUC improvement : +{auc_aug - auc_orig:.4f}')
print('\nDetailed report (GAN-Augmented model):')
print(classification_report(true_labels, aug_preds, target_names=['Normal', 'Pneumonia']))